# The bench store: what it holds

`tests/e2e/bench/sync.mjs` pushes every sweep to Garage on mlbox: the score rows, model calls, price snapshots and box
frames as Parquet (deduplicated by the store's own `views.sql`), and each run's folder as plain files under
`traces/<sweep>/`. `wmlnb.store()` opens it as DuckDB with those views loaded. This notebook is the inventory, and the
pattern to copy: query the views inside `wmlnb.extract`, so the numbers stay pinned while the store keeps growing.

In [1]:
import pandas as pd
import wmlnb

HERE = wmlnb.ROOT / "notebooks" / "bench"
DATA = HERE / "data" / "store-tour"
pd.set_option("display.width", 160)

_con = None
def con():
    """The store, opened only when an extract has to read it (a pinned run never connects)."""
    global _con
    _con = _con or wmlnb.store()
    return _con

## The rows, pinned

One row per sweep and model from the `scores` view, and the number of run folders each sweep has under `traces/`.

In [2]:
BY_MODEL = '''
SELECT sweep, model, local::BOOLEAN AS local, count(*) AS runs,
       sum(CASE WHEN scored = 1 THEN 1 ELSE 0 END) AS scored, sum(CASE WHEN passed = 1 THEN 1 ELSE 0 END) AS passed,
       sum(tokens) AS tokens, round(sum(secs) / 60, 1) AS minutes,
       min("at") AS first_at, max("at") AS last_at, count(DISTINCT "by") AS clones
FROM scores GROUP BY ALL ORDER BY sweep, model'''

def traces():
    bucket = con().sql("SELECT getvariable('wml_bucket')").fetchone()[0]
    q = f"SELECT split_part(file, '/', 5) AS sweep, count(*) AS run_dirs FROM glob('s3://{bucket}/traces/**/cell.json') GROUP BY 1 ORDER BY 1"
    return con().sql(q).df().to_dict("records")

by_model = pd.DataFrame(wmlnb.extract(DATA / "by-model.json", lambda: con().sql(BY_MODEL).df().astype({"first_at": str, "last_at": str}).to_dict("records"),
                                      source={"store": "bench store (sync.mjs)", "view": "scores", "query": BY_MODEL.strip()}))
trace_dirs = pd.DataFrame(wmlnb.extract(DATA / "traces.json", traces, source={"store": "bench store (sync.mjs)", "glob": "traces/**/cell.json"}))
pd.DataFrame(wmlnb.provenance(DATA / "by-model.json", DATA / "traces.json"))

,extract,sha256,rows,extracted_at,commit,query,store,view,glob
0,notebooks/bench/data/store-tour/by-model.json,0896eb42e3e664ec,78,2026-10-13T21:32:12+00:00,71f0d3258b72,"SELECT sweep, model, local::BOOLEAN AS local, ...",bench store (sync.mjs),scores,NaN
1,notebooks/bench/data/store-tour/traces.json,5e4365c67c97d1fe,19,2026-10-13T21:32:13+00:00,71f0d3258b72,NaN,bench store (sync.mjs),NaN,traces/**/cell.json


## Sweeps

Runs and models per sweep, how many runs carry a pass/fail score (an interview has none: it is read, not scored), and
whether the run folders are in the store as well as the rows.

In [3]:
sweeps = (by_model.groupby("sweep")
          .agg(runs=("runs", "sum"), models=("model", "nunique"), local_models=("local", "sum"), scored=("scored", "sum"),
               passed=("passed", "sum"), minutes=("minutes", "sum"), first=("first_at", "min"), last=("last_at", "max"))
          .join(trace_dirs.set_index("sweep"), how="outer").fillna({"run_dirs": 0})
          .sort_values("last", ascending=False))
sweeps = sweeps.fillna({"runs": 0, "models": 0, "local_models": 0, "scored": 0, "passed": 0})
sweeps["first"] = sweeps["first"].str[:10]
sweeps["last"] = sweeps["last"].str[:10]
sweeps.astype({c: int for c in ["runs", "models", "local_models", "scored", "passed", "run_dirs"]})

,runs,models,local_models,scored,passed,minutes,first,last,run_dirs
sweep,,,,,,,,,
panel-shell-shapes,8,8,8,0,0,37.0,2026-10-13,2026-10-13,8
cuts2-python-fix-glm,32,1,1,32,18,9.4,2026-10-10,2026-10-10,32
cuts2-python-fix,32,1,0,32,32,9.6,2026-10-10,2026-10-10,32
cuts2-recheck2,193,3,2,193,148,39.4,2026-10-10,2026-10-10,193
cuts2-recheck,188,3,2,188,150,46.1,2026-10-10,2026-10-10,0
prompt-budget-cuts2,264,11,3,264,244,97.5,2026-10-10,2026-10-10,264
cuts2-pilot,10,1,0,10,8,2.2,2026-10-10,2026-10-10,10
tool-details-v2,330,11,3,330,318,97.5,2026-10-10,2026-10-10,330
tool-details-navfix,143,3,2,143,117,24.3,2026-10-09,2026-10-09,144


A sweep with `run_dirs` but no `runs` has folders in the store that no clone's score log has pushed yet. The reverse
(rows, no folders) is a sweep pushed with `--only-db`, one whose push of its folders failed, or one whose folders the
push does not recognise: `cuts2-recheck` (2026-10-13 extract), which `sync.mjs status` counts as "0 of 0 runs".

## Models

Each model across every sweep: how much of the store is about it.

In [4]:
(by_model.groupby("model")
 .agg(sweeps=("sweep", "nunique"), runs=("runs", "sum"), local=("local", "max"), scored=("scored", "sum"),
      passed=("passed", "sum"), tokens=("tokens", "sum"), minutes=("minutes", "sum"))
 .assign(pass_rate=lambda d: (d.passed / d.scored).where(d.scored > 0).round(2))
 .sort_values("runs", ascending=False))

,sweeps,runs,local,scored,passed,tokens,minutes,pass_rate
model,,,,,,,,
glm-4.7-flash:latest,11,314,True,312.0,152.0,19074419.0,83.4,0.49
openrouter.minimax/minimax-m3,10,308,False,307.0,280.0,15209691.0,91.4,0.91
qwen3.6:35b,9,282,True,281.0,252.0,13899805.0,40.8,0.90
litellm.anthropic/claude-sonnet-5-5,5,109,False,109.0,103.0,5442038.0,26.7,0.94
litellm.google/gemini-3.6-flash,6,105,False,100.0,94.0,5112057.0,27.5,0.94
deepseek.deepseek-v4-pro,6,100,False,95.0,94.0,5913089.0,48.6,0.99
openrouter.openai/gpt-6-luna,6,99,False,94.0,93.0,3712382.0,31.0,0.99
gemma4:31b,7,92,True,85.0,85.0,4138722.0,47.3,1.00
deepseek.deepseek-flash,3,86,False,86.0,86.0,3749426.0,24.0,1.00


## One sweep: the shell-shapes interview on local models

`panel-shell-shapes` (2026-10-13) put the `ml.sh` API-shape interview to eight Ollama models
(`tests/e2e/panel/shell-shapes.md` has the scored results). What the store's rows add is the cost: tokens and minutes
per model for the same eight turns. Tokens count every call's whole prompt again (the interview's history grows each
turn), so they measure context carried, not text written.

In [5]:
(by_model[by_model.sweep == "panel-shell-shapes"][["model", "tokens", "minutes"]]
 .astype({"tokens": int}).sort_values("minutes").reset_index(drop=True))

,model,tokens,minutes
0,nemotron-3-nano:4b,145288,0.6
1,granite4.1:3b,105011,0.8
2,qwen3.5:4b,122499,2.7
3,qwen3.8:27b,142855,5.0
4,qwen3.5:9b,143693,6.0
5,glm-4.7-flash:latest,135874,6.6
6,gemma4:31b,122937,7.5
7,gemma4:12b,165587,7.8
